# EXP26: $3000 资本全维度分析

1. RISK_PER_TRADE 从 $50(2.5%) 到 $75(2.5%) 或 $100(3.3%)
2. 不同风险比例下的 Sharpe/MaxDD/破产概率
3. 最大手数扫描

In [ ]:
import sys; sys.path.insert(0, '..')
from backtest import DataBundle, run_variant, C12_KWARGS
import pandas as pd
import numpy as np

data = DataBundle.load_default()

LIVE_KWARGS = {
    **C12_KWARGS,
    "intraday_adaptive": True,
    "choppy_threshold": 0.35,
    "kc_only_threshold": 0.60,
    "spread_cost": 0.50,
}
print('Ready')

In [ ]:
# Part 1: Baseline at current $50 risk
baseline = run_variant(data, "Baseline_$50risk", **LIVE_KWARGS)
trades = baseline['_trades']
pnls = np.array([t.pnl for t in trades])
equity = np.cumsum(pnls)
max_dd = (equity - np.maximum.accumulate(equity)).min()

print(f"Baseline ($50 risk, 0.01-0.03 lot):")
print(f"  PnL=${pnls.sum():.0f}, MaxDD=${max_dd:.0f}, N={len(pnls)}")
print(f"  DD/Capital: $2000={abs(max_dd)/2000*100:.1f}%, $3000={abs(max_dd)/3000*100:.1f}%")

In [ ]:
# Part 2: 不同资本下的风险指标
np.random.seed(42)
n_sims = 5000

print("\n=== Risk Metrics by Capital and Risk Level ===")
print(f"{'Capital':>8s} {'Risk$':>6s} {'Risk%':>6s} {'DD/Cap':>7s} {'Ruin%':>6s} {'MaxLot':>7s}")

for capital in [2000, 2500, 3000, 3500]:
    for risk_pct in [0.020, 0.025, 0.030, 0.033, 0.040]:
        risk_usd = capital * risk_pct
        # Scale PnLs proportionally to risk
        scale = risk_usd / 50.0  # baseline uses $50
        scaled_pnls = pnls * scale
        scaled_eq = np.cumsum(scaled_pnls)
        scaled_dd = (scaled_eq - np.maximum.accumulate(scaled_eq)).min()
        
        # Ruin probability
        ruin = 0
        for _ in range(n_sims):
            shuffled = np.random.permutation(scaled_pnls)
            eq = capital + np.cumsum(shuffled)
            if eq.min() <= 0:
                ruin += 1
        ruin_pct = ruin / n_sims * 100
        
        # Max lot at this risk level (ATR ~$15-25 for gold)
        avg_atr_usd = 20  # rough avg
        max_lot = risk_usd / (4.5 * avg_atr_usd * 100)  # SL=4.5ATR, 100 oz/lot
        
        dd_pct = abs(scaled_dd) / capital * 100
        print(f"${capital:>6d} ${risk_usd:>5.0f} {risk_pct:>5.1%} {dd_pct:>6.1f}% {ruin_pct:>5.1f}% {max_lot:>6.3f}")

In [ ]:
# Part 3: 最优 risk% 推荐
print("\n=== Recommendation for $3000 Capital ===")
for risk_pct in [0.020, 0.025, 0.030, 0.033]:
    risk_usd = 3000 * risk_pct
    scale = risk_usd / 50.0
    scaled_pnls = pnls * scale
    total = scaled_pnls.sum()
    scaled_eq = np.cumsum(scaled_pnls)
    scaled_dd = abs((scaled_eq - np.maximum.accumulate(scaled_eq)).min())
    sharpe_proxy = scaled_pnls.mean() / scaled_pnls.std() * np.sqrt(252*6) if scaled_pnls.std() > 0 else 0
    
    ruin = 0
    for _ in range(n_sims):
        shuffled = np.random.permutation(scaled_pnls)
        eq = 3000 + np.cumsum(shuffled)
        if eq.min() <= 0:
            ruin += 1
    
    print(f"  Risk {risk_pct:.1%} (${risk_usd:.0f}/trade): "
          f"PnL=${total:.0f}, MaxDD=${scaled_dd:.0f} ({scaled_dd/3000*100:.1f}%), "
          f"Ruin={ruin/n_sims*100:.1f}%, Sharpe~{sharpe_proxy:.2f}")

In [ ]:
import json
with open('../data/exp26_results.json', 'w') as f:
    json.dump({
        'baseline_pnl': float(pnls.sum()),
        'baseline_max_dd': float(max_dd),
        'baseline_n': len(pnls),
        'note': '$3000 capital analysis, risk scaling from $50 baseline'
    }, f, indent=2)
print('Saved')